In [ ]:
import sqlite3
import pandas as pd
#TABELE DO ZADAŃ 5 - 7:

#customers
df_customers =  pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Zając'],
    'city': ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
})

#orders
df_orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 1, 3, 1],
    'amount': [250.0, 120.5, 340.0, 80.0, 190.0]
})

#products
df_products = pd.DataFrame({
'product_id': [1, 2, 3, 4, 5],
'name': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'],
'category': ['Elektronika', 'Akcesoria', 'Akcesoria', 'Elektronika', 'Akcesoria'],
'price': [3500, 50, 200, 1200, 150],
'stock': [10, 150, 80, 25, 60]
})
conn = sqlite3.connect(':memory:')
df_customers.to_sql('customers', conn, index=False)
df_orders.to_sql('orders', conn, index=False)
df_products.to_sql('products', conn, index=False)



5

##ZADANIE 5

In [ ]:
query1 = """
SELECT
  name, order_id
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
"""

query2 = """
SELECT
  name, COALESCE(o.amount, 0) as amount
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
"""

query3 = """
SELECT
  name, order_id
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
WHERE order_id IS NULL
"""

print(pd.read_sql_query(query1,conn))
print(pd.read_sql_query(query2,conn))
print(pd.read_sql_query(query3,conn))

               name  order_id
0      Jan Kowalski     101.0
1      Jan Kowalski     103.0
2      Jan Kowalski     105.0
3        Anna Nowak     102.0
4  Piotr Wiśniewski     104.0
5       Maria Zając       NaN
               name  amount
0      Jan Kowalski   190.0
1      Jan Kowalski   250.0
2      Jan Kowalski   340.0
3        Anna Nowak   120.5
4  Piotr Wiśniewski    80.0
5       Maria Zając     0.0
          name order_id
0  Maria Zając     None


## Zadanie 6

In [ ]:
query4 = """
SELECT
  name,
  COUNT(o.customer_id) as liczba_zamowien
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
GROUP BY name
"""

query5 = """
SELECT
  name, SUM(o.amount) as suma
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
GROUP BY name
"""

query6 = """
SELECT
  name, AVG(o.amount) as srednia
FROM
  customers c
LEFT JOIN orders o ON c.customer_id=o.customer_id
GROUP BY name
"""

print(pd.read_sql_query(query4,conn))
print(pd.read_sql_query(query5,conn))
print(pd.read_sql_query(query6,conn))

               name  liczba_zamowien
0        Anna Nowak                1
1      Jan Kowalski                3
2       Maria Zając                0
3  Piotr Wiśniewski                1
               name   suma
0        Anna Nowak  120.5
1      Jan Kowalski  780.0
2       Maria Zając    NaN
3  Piotr Wiśniewski   80.0
               name  srednia
0        Anna Nowak    120.5
1      Jan Kowalski    260.0
2       Maria Zając      NaN
3  Piotr Wiśniewski     80.0


## ZADANIE 7

In [ ]:
query7 = """
SELECT
  COUNT(product_id) as liczba_produktow
FROM
  products
"""

query8 = """
SELECT
  SUM(stock) as suma_produktow
FROM
  products
"""

query9 = """
SELECT
  category,
  AVG(price) as srednia_cena
FROM
  products
GROUP BY category
"""

print(pd.read_sql_query(query7,conn))
print(pd.read_sql_query(query8,conn))
print(pd.read_sql_query(query9,conn))

   liczba_produktow
0                 5
   suma_produktow
0             325
      category  srednia_cena
0    Akcesoria    133.333333
1  Elektronika   2350.000000


## ZADANIE 8

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import time

np.random.seed(5)

conn = sqlite3.connect(':memory:')

large_data = pd.DataFrame({
'user_id': range(1, 10001),
'name': [f'User_{i}' for i in range(1, 10001)],
'age': np.random.randint(18, 80, 10000),
'city': np.random.choice(['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław', 'Poznań'], 10000),
'registration_date': pd.date_range('2020-01-01', periods=10000, freq='15min')
})

large_data.to_sql('data', conn, index=False)
query = "SELECT * FROM data WHERE age > 60 ORDER BY registration_date DESC LIMIT 100"

plan_before = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("Plan wykonania BEZ indeksu:")
print(plan_before)

# Pomiar czasu (uruchamiamy kilka razy dla stabilności i uśredniamy)
durations = []
for _ in range(5):
  start = time.perf_counter()
  result_before = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_before = sum(durations) / len(durations)
print(f"\nŚredni czas BEZ indeksu: {time_before*1000:.2f} ms")

# === Test 2: Z indeksem ===
conn.execute("CREATE INDEX idx_age ON data(age)")
conn.execute("CREATE INDEX idx_registration_date ON data(registration_date)")

# Plan wykonania
plan_after = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("\nPlan wykonania Z indeksem:")
print(plan_after)

# Pomiar czasu
durations = []
for _ in range(5):
  start = time.perf_counter()
  result_after = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_after = sum(durations) / len(durations)
print(f"\nŚredni czas Z indeksem: {time_after*1000:.2f} ms")
print(f"Przyspieszenie: {time_before / time_after:.1f}x")

# Wyświetlenie wyników
print("\nPierwsze 5 wyników zapytania:")
print(result_after.head())
conn.close()


Plan wykonania BEZ indeksu:
   id  parent  notused                        detail
0   4       0        0                     SCAN data
1  21       0        0  USE TEMP B-TREE FOR ORDER BY

Średni czas BEZ indeksu: 4.47 ms

Plan wykonania Z indeksem:
   id  parent  notused                                       detail
0   5       0        0  SCAN data USING INDEX idx_registration_date

Średni czas Z indeksem: 1.09 ms
Przyspieszenie: 4.1x

Pierwsze 5 wyników zapytania:
   user_id       name  age      city    registration_date
0     9998  User_9998   66   Wrocław  2020-04-14 03:15:00
1     9995  User_9995   74  Warszawa  2020-04-14 02:30:00
2     9994  User_9994   73    Kraków  2020-04-14 02:15:00
3     9993  User_9993   73    Poznań  2020-04-14 02:00:00
4     9992  User_9992   76  Warszawa  2020-04-14 01:45:00


## Zadanie 18

In [23]:
import sqlite3
import pandas as pd
import numpy as np
import time
np.random.seed(2)

conn = sqlite3.connect(':memory:')


employee_data = pd.DataFrame({
'employee_id': range(1, 21),
'name': [f'Employee_{i}' for i in range(1, 21)],
'manager_id': ['None' if i<6 else np.random.randint(1, 6) for i in range(1,21)]
})

employee_data.to_sql('employees', conn, index=False)

query18 = """
SELECT
 e1.name AS employee,
 e2.name AS manager
FROM employees e1
LEFT JOIN employees e2
  ON e1.manager_id = e2.employee_id
"""

query19="""
SELECT
  e.name AS employee
FROM employees e
WHERE manager_id IS 'None'
"""

query20="""
SELECT
  manager_id,
  COUNT(name) as liczba_pracownikow
FROM employees
WHERE manager_id IS NOT 'None'
GROUP BY manager_id
"""

print(pd.read_sql_query(query18,conn))
print(pd.read_sql_query(query19,conn))
print(pd.read_sql_query(query20,conn))

       employee     manager
0    Employee_1        None
1    Employee_2        None
2    Employee_3        None
3    Employee_4        None
4    Employee_5        None
5    Employee_6  Employee_1
6    Employee_7  Employee_1
7    Employee_8  Employee_4
8    Employee_9  Employee_3
9   Employee_10  Employee_4
10  Employee_11  Employee_1
11  Employee_12  Employee_3
12  Employee_13  Employee_2
13  Employee_14  Employee_4
14  Employee_15  Employee_3
15  Employee_16  Employee_5
16  Employee_17  Employee_5
17  Employee_18  Employee_5
18  Employee_19  Employee_4
19  Employee_20  Employee_5
     employee
0  Employee_1
1  Employee_2
2  Employee_3
3  Employee_4
4  Employee_5
  manager_id  liczba_pracownikow
0          1                   3
1          2                   1
2          3                   3
3          4                   4
4          5                   4
